In [3]:
import os
import shutil
from datetime import date
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip


In [4]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, TimestampType, DateType,
    BooleanType, DecimalType,
)

SCHEMAS = {
    "reproducoes": StructType([
        StructField("id",            IntegerType(),  False),
        StructField("usuario_id",    IntegerType(),  False),
        StructField("musica_id",     IntegerType(),  False),
        StructField("dispositivo_id",IntegerType(),  False),
        StructField("timestamp",     TimestampType(),False),
        StructField("ms_tocados",    IntegerType(),  False),
        StructField("completou",     BooleanType(),  False),
        StructField("created_at",    TimestampType(),False),
    ]),
    "pagamentos": StructType([
        StructField("id",            IntegerType(),   False),
        StructField("assinatura_id", IntegerType(),   False),
        StructField("valor",         DecimalType(8,2),False),
        StructField("data",          DateType(),      False),
        StructField("status",        StringType(),    False),
        StructField("created_at",    TimestampType(), False),
    ]),
    "assinaturas": StructType([
        StructField("id",          IntegerType(),   False),
        StructField("usuario_id",  IntegerType(),   False),
        StructField("plano_id",    IntegerType(),   False),
        StructField("data_inicio", DateType(),      False),
        StructField("data_fim",    DateType(),      True),
        StructField("status",      StringType(),    False),
        StructField("created_at",  TimestampType(), False),
    ]),
}


In [5]:
load_dotenv()

is_docker = os.path.exists('/.dockerenv')

MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

builder = (
    SparkSession.builder
    .appName("bronze-fatos")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

cliente_minio = Minio(
    MINIO_HOST,
    access_key=MINIO_USER,
    secret_key=MINIO_PASS,
    secure=False,
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 16:58:37 WARN Utils: Your hostname, DESKTOP-S3JB25M, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/21 16:58:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/c/Users/felip/www/university/eng-dados/projeto-ed-final/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/felipe/.ivy2.5.2/cache
The jars for the packages stored in: /home/felipe/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8b34ad27-fd0f-42de-b826-2b23c6017ad8;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf

In [6]:
buckets = {b.name for b in cliente_minio.list_buckets()}
print("Buckets disponíveis:", buckets)

if "bronze" not in buckets:
    cliente_minio.make_bucket("bronze")
    print("Bucket 'bronze' criado.")


Buckets disponíveis: {'landing', 'bronze', 'gold', 'silver'}


In [7]:
def download_landing_csv(tabela: str, cliente: Minio, destino_dir: Path) -> Path:
    """Baixa o CSV da Landing (MinIO) para um diretório local."""
    destino_dir.mkdir(parents=True, exist_ok=True)
    destino_arquivo = destino_dir / f"{tabela}.csv"
    cliente.fget_object("landing", f"{tabela}/{tabela}.csv", str(destino_arquivo))
    return destino_arquivo


def upload_delta_para_minio(local_dir: Path, bucket: str, prefix: str,
                            cliente: Minio) -> None:
    """Sobe recursivamente a estrutura Delta (parquet + _delta_log) para o MinIO."""
    for arq in sorted(local_dir.rglob("*")):
        if arq.is_file():
            object_name = f"{prefix}/{arq.relative_to(local_dir)}"
            cliente.fput_object(bucket, object_name, str(arq))
            print(f"  → {object_name}")


In [8]:
INGESTAO_DATE = str(date.today())
print(f"ingestao_date = {INGESTAO_DATE}")


ingestao_date = 2026-06-21


In [ ]:
for tabela, schema in SCHEMAS.items():
    print(f"\n{'=' * 55}")
    print(f"Tabela: {tabela}")
    print(f"{'=' * 55}")

    local_csv_dir = Path(f"/tmp/landing_local/{tabela}")
    csv_path = download_landing_csv(tabela, cliente_minio, local_csv_dir)
    print(f"CSV baixado: {csv_path}")

    df = (
        spark.read
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss.SSSSSS")
        .option("dateFormat", "yyyy-MM-dd")
        .schema(schema)
        .csv(str(csv_path))
    )

    total = df.count()
    print(f"Registros lidos: {total:,}")

    df = df.withColumn("ingestao_date", F.lit(INGESTAO_DATE).cast("date"))

    local_delta_dir = Path(f"/tmp/bronze/{tabela}")
    if local_delta_dir.exists():
        shutil.rmtree(local_delta_dir)

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("ingestao_date")
        .save(str(local_delta_dir))
    )
    print(f"Delta escrito localmente em {local_delta_dir}")

    print(f"Enviando para MinIO bronze/fatos/{tabela}/")
    upload_delta_para_minio(
        local_delta_dir, "bronze", f"fatos/{tabela}", cliente_minio
    )
    print(f"{tabela} concluído — {total:,} registros na Bronze.")

print("\nPromoção Landing → Bronze concluída para todas as tabelas fato.")

In [10]:
spark.stop()
